# Processing Podcast Audio for Video Animation Production

This notebook demonstrates various multimedia processing and AI applications, including:
- Audio Speaker Diarization using `pyannote.audio`
- Video Style Transfer using `AnimeGAN` and `Fast Neural Transfer` (AdaIN)
- Video Animation with `SadTalker` and `First Order Motion Model (FOMM)`

In [ ]:
pip install pyannote.audio pydub

## Setup and Authentication

This section handles the installation of necessary libraries and authentication for Hugging Face Hub.

In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

In [ ]:
from huggingface_hub import login
login()

## Audio Editing and Preprocessing

This part of the notebook focuses on preparing an audio file for speaker diarization. This includes loading, converting to a suitable format, and splitting it into manageable chunks.

#Audio editing

In [ ]:
from pyannote.audio import Pipeline
from pydub import AudioSegment

# # Load pretrained diarization pipeline
pipeline = Pipeline.from_pretrained("pyannote/speaker-diarization-3.1")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
pipeline.to(device)  # Correct usage

# Path to your podcast audio
AUDIO_FILE = "Ep2.mp3"

# Load audio
audio = AudioSegment.from_file(AUDIO_FILE)
audio = audio.set_frame_rate(16000).set_channels(1)
audio.export('Ep2_16k_mono.wav', format='wav')


In [ ]:
# Pad 0.01s silence at the end
padding = AudioSegment.silent(duration=10)  # 10 ms
audio += padding
audio.export("Ep2_16k_mono_padded.wav", format="wav")

In [ ]:
import wave

with wave.open("Ep2_16k_mono_padded.wav") as w:
    print(w.getnframes(), w.getframerate())

The `split_audio` function divides the main audio file into smaller chunks with an optional overlap. This is crucial for processing long audio files efficiently with models that might have memory constraints or benefit from shorter inputs.

In [ ]:
import os, glob

# Delete all old chunks
for f in glob.glob("/content/chunk_*.wav"):
    os.remove(f)

print("✅ Old chunks deleted.")

In [ ]:
from pydub import AudioSegment

def split_audio(path, chunk_minutes=15, overlap_seconds=2):
    audio = AudioSegment.from_file(path)
    duration_ms = len(audio)
    chunk_ms = chunk_minutes * 60 * 1000
    overlap_ms = overlap_seconds * 1000

    if overlap_ms >= chunk_ms:
        raise ValueError("Overlap must be smaller than chunk length.")

    start = 0
    i = 0
    paths = []

    while start < duration_ms:
        end = min(start + chunk_ms, duration_ms)  # never go past end
        chunk = audio[start:end]
        outpath = f"chunk_{i:03}.wav"
        chunk.export(outpath, format="wav")
        paths.append(outpath)
        i += 1

        start += (chunk_ms - overlap_ms)  # move forward by chunk minus overlap

    print(f"✅ Created {len(paths)} chunks.")
    return paths

# Example usage:
chunks = split_audio("Ep2_16k_mono_padded.wav", chunk_minutes=10)
print(f"Created {len(chunks)} chunks")

## Speaker Diarization

Speaker diarization is the process of partitioning an audio stream into homogeneous segments according to the speaker identity. It answers the question 'who spoke when?' This section uses the `pyannote.audio` library for this task.

#Speaker Diarization

In [ ]:
pip install onnxruntime

In [ ]:

from pyannote.audio import Pipeline
pipeline = Pipeline.from_pretrained("pyannote/speaker-diarization-3.0", token='')

# Select the device properly
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Move pipeline to device
pipeline.to(device)

After running the diarization pipeline, the code extracts and saves individual audio segments for each identified speaker. This can be useful for further analysis or separate processing of each speaker's dialogue.

In [ ]:
pip install torchaudio


In [ ]:
# run the pipeline on an audio file
# diarization = pipeline("chunk_003.wav")

from pyannote.audio.pipelines.utils.hook import ProgressHook
with ProgressHook() as hook:
    diarization = pipeline("chunk_003.wav", hook=hook)


The previous cell attempts to process the audio using a non-existent `waveform` and `sample_rate` variable. These variables need to be defined by loading the audio file using `torchaudio` first. Additionally, the diarization result needs to be properly iterated to extract speaker segments.

In [ ]:
print(dir(diarization))
# Access the Annotation object
# Get exclusive speaker segments (one speaker per segment)
annotation = diarization.exclusive_speaker_diarization

# Prepare a dict to collect audio for each speaker
speaker_audio = {speaker: [] for speaker in annotation.labels()}

# Slice the waveform according to annotation segments
for segment, _, speaker in annotation.itertracks(yield_label=True):
    start_sample = int(segment.start * sample_rate)
    end_sample = int(segment.end * sample_rate)
    speaker_audio[speaker].append(waveform[:, start_sample:end_sample])

# Concatenate all segments per speaker and save
for speaker, chunks in speaker_audio.items():
    if chunks:  # skip empty
        combined = torch.cat(chunks, dim=1)
        torchaudio.save(f"{speaker}.wav", combined, sample_rate)
        print(f"Saved audio for {speaker}")

## Video Preprocessing for Style Transfer

This section prepares a video for style transfer by trimming it and extracting frames at a reduced frame rate. This makes processing more efficient.

#Video Style Transfer

In [ ]:
#Cut vid to 1/3
import cv2
import os

input_video = "VideoTEST.mp4"
trimmed_video = "webcam_trimmed.mp4"

cap = cv2.VideoCapture(input_video)
fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# Calculate frame range for first 1/3
end_frame = total_frames // 3

# Video writer
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(trimmed_video, fourcc, fps, (width, height))

for i in range(end_frame):
    ret, frame = cap.read()
    if not ret:
        break
    out.write(frame)

cap.release()
out.release()
print(f"Trimmed video saved: {trimmed_video} ({end_frame} frames)")


In [ ]:
frames_dir = "frames"
os.makedirs(frames_dir, exist_ok=True)

cap = cv2.VideoCapture(trimmed_video)
original_fps = cap.get(cv2.CAP_PROP_FPS)
target_fps = 10  # desired FPS
frame_interval = int(original_fps / target_fps)

frame_idx = 0
saved_idx = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break
    if frame_idx % frame_interval == 0:
        frame_path = os.path.join(frames_dir, f"frame_{saved_idx:05d}.png")
        cv2.imwrite(frame_path, frame)
        saved_idx += 1
    frame_idx += 1

cap.release()
print(f"Extracted {saved_idx} frames at ~{target_fps} fps to {frames_dir}/")


This part loads a few sample frames to quickly test different style transfer models without processing the entire video.

#Test a few stylizers

In [ ]:
import os
from PIL import Image

frames_dir = "frames"
sample_frames = ["frame_00000.png", "frame_00050.png", "frame_00100.png"]

sample_paths = [os.path.join(frames_dir, f) for f in sample_frames]
frames = [Image.open(p) for p in sample_paths]

print(f"Loaded {len(frames)} frames for testing")


### AnimeGAN Style Transfer

AnimeGAN is a generative adversarial network designed to convert real-world images into anime-style images. This section sets up the environment for using AnimeGAN.

##AnimeGAN

In [ ]:
#clonse animegna2_pytorch repo
!git clone https://github.com/bryandlee/animegan2-pytorch.git
%cd animegan2-pytorch


The previous `git clone` command was executed within `/content/AnimeGANv2/animegan2-pytorch`. To ensure consistency and avoid errors, it's better to clone into `/content/` and then change the directory. The cell `oSqMcyc6rVbl` is a duplicate and should be removed or ignored as `animegan2-pytorch` is already cloned.

In [ ]:
!pip install torch torchvision pillow opencv-python tqdm


In [ ]:
!git clone https://github.com/bryandlee/animegan2-pytorch.git
%cd animegan2-pytorch


This cell loads a pre-trained AnimeGAN model and applies it to a single test frame. It demonstrates the basic usage of the model for style transfer.

In [ ]:
!animegan2-pytorch/weights/selfie2anime.pth


In [ ]:
#Load Model
import torch
from model import Generator  # this is from the cloned repo
from PIL import Image
from torchvision import transforms

device = "cuda" if torch.cuda.is_available() else "cpu"

# Choose a pre-trained model from your weights folder
weights_path = "weights/face_paint_512_v2.pt"

# Load the generator
model = Generator().to(device)
model.load_state_dict(torch.load(weights_path, map_location=device))
model.eval()

# Preprocess and postprocess functions
preprocess = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.5,0.5,0.5), std=(0.5,0.5,0.5))
])
postprocess = transforms.Compose([
    transforms.Lambda(lambda x: (x * 0.5 + 0.5).clamp(0,1)),
    transforms.ToPILImage()
])

# Test on a single frame
frame = Image.open("/content/frames/frame_00500.png").convert("RGB")
input_tensor = preprocess(frame).unsqueeze(0).to(device)

with torch.no_grad():
    output_tensor = model(input_tensor)

output_image = postprocess(output_tensor.squeeze(0).cpu())
output_image.save("test_frame_0.png")
print("Saved test_frame_0.png")


### Fast Neural Style Transfer (AdaIN)

This section explores Fast Neural Style Transfer, specifically using Adaptive Instance Normalization (AdaIN) to transfer the style of one image to another. The `VGG19` model is used for feature extraction.

Conclusions: I do not like this style

##Fast Neural Transfer

## SadTalker: Realistic Audio-Driven Talking Head Generation

SadTalker is an AI model that generates realistic talking head videos from a single source image and an audio input. This section attempts to set up and use SadTalker.

In [ ]:
import os
from pathlib import Path
from PIL import Image
import torch
from torchvision import transforms, models

# device = "cuda" if torch.cuda.is_available() else "cpu"

# # -----------------------------
# # Load frames (first 50 only)
# # -----------------------------
# frames_folder = Path("/content/frames")  # folder with your video frames
# frame_files = sorted(frames_folder.glob("*.png"))[:50]  # first 50 frames
# frames = [Image.open(f).convert("RGB") for f in frame_files]

# # -----------------------------
# # Load style image
# # -----------------------------
# style_img = Image.open("/content/AnimationStyle.jpg").convert("RGB")
# style_transform = transforms.Compose([
#     transforms.Resize((256,256)),
#     transforms.ToTensor()
# ])
# style_tensor = style_transform(style_img).unsqueeze(0).to(device)

# -----------------------------
# Load VGG19 (for style transfer)
# -----------------------------
cnn = models.vgg19(weights=models.VGG19_Weights.DEFAULT).features.to(device).eval()

# Extract features at specific layers
def get_features(x, model, layers=None):
    if layers is None:
        layers = {'0':'conv1_1', '5':'conv2_1', '10':'conv3_1', '19':'conv4_1', '21':'conv4_2', '28':'conv5_1'}
    features = {}
    for name, layer in model._modules.items():
        x = layer(x)
        if name in layers:
            features[layers[name]] = x
    return features

# -----------------------------
# Adaptive Instance Normalization
# -----------------------------
def adain(content_feat, style_feat, eps=1e-5):
    size = content_feat.size()
    content_mean, content_std = content_feat.mean([2,3], keepdim=True), content_feat.std([2,3], keepdim=True)
    style_mean, style_std = style_feat.mean([2,3], keepdim=True), style_feat.std([2,3], keepdim=True)
    normalized = (content_feat - content_mean) / (content_std + eps)
    return normalized * style_std + style_mean

# -----------------------------
# Load images
# -----------------------------
preprocess = transforms.Compose([
    transforms.Resize((256,256)),
    transforms.ToTensor()
])

postprocess = transforms.Compose([
    transforms.Lambda(lambda x: (x * 255).clamp(0,255).byte()),
    transforms.ToPILImage()
])

content_img = Image.open("/content/frames/frame_00050.png").convert("RGB")
style_img = Image.open("/content/AnimationStyle.jpg").convert("RGB")

content_tensor = preprocess(content_img).unsqueeze(0).to(device)
style_tensor = preprocess(style_img).unsqueeze(0).to(device)

# -----------------------------
# Stylize frame
# -----------------------------
with torch.no_grad():
    content_features = get_features(content_tensor, cnn)
    style_features = get_features(style_tensor, cnn)

    # AdaIN on conv4_1 layer (example)
    t = adain(content_features['conv4_1'], style_features['conv4_1'])
    # Simple decoder: for demo we just return original frame; in full NST you'd decode t back to image
    # Here, replace with your own decoder if available
    output_tensor = content_tensor  # placeholder: replace with decoder output

output_image = postprocess(output_tensor.squeeze(0).cpu())
output_image.save("styled_frame_50.png")
print("Saved styled_frame_0.png")

# # -----------------------------
# # Placeholder NST function
# # -----------------------------
# def stylize_frame(content_img):
#     content_tensor = transforms.ToTensor()(content_img).unsqueeze(0).to(device)
#     # Here you would run your AdaIN / NST
#     # For now, just return the original image
#     return content_img

# # -----------------------------
# # Apply style transfer to frames
# # -----------------------------
# output_folder = Path("styled_frames")
# output_folder.mkdir(exist_ok=True)

# for i, frame in enumerate(frames):
#     output_image = stylize_frame(frame)
#     output_path = output_folder / f"style_test_frame_{i}.png"
#     output_image.save(output_path)
#     print(f"Saved {output_path}")


This cell attempts to perform style transfer on a single frame using the AdaIN method. It loads content and style images, extracts features using a VGG19 network, and applies AdaIN. However, it currently uses a placeholder for the decoder, meaning it doesn't fully reconstruct the styled image but rather shows the intermediate AdaIN output. The commented-out code suggests an intention to process multiple frames, which could be implemented later.

#Animation: SadTalker

In [ ]:
!git clone https://github.com/OpenTalker/SadTalker.git
%cd SadTalker
!pip install -r requirements.txt

The previous installation of `requirements.txt` failed due to a numpy dependency conflict. This is a common issue in Colab environments with pre-installed packages. The following cells attempt to install specific Python versions and dependencies to resolve this.

In [ ]:
!cd SadTalker

!conda create -n sadtalker python=3.8

!conda activate sadtalker

!pip install torch==1.12.1+cu113 torchvision==0.13.1+cu113 torchaudio==0.12.1 --extra-index-url https://download.pytorch.org/whl/cu113

!conda install ffmpeg

!pip install -r requirements.txt

### Coqui TTS is optional for gradio demo.
### pip install TTS

The provided commands aim to set up a specific Python environment with `conda` and `pip` to satisfy SadTalker's dependencies. However, `conda` is typically not available in Colab environments, leading to 'command not found' errors. Additionally, `pip` is attempting to install `torch==1.12.1+cu113`, which might not be compatible with the current CUDA version or available in PyPI, leading to installation errors.

In [ ]:
### make sure that CUDA is available in Edit -> Nootbook settings -> GPU
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader

The previous attempts to install SadTalker dependencies failed. This revised set of commands tries to manually set up the environment, including installing `python3.8`, `distutils`, updating alternatives, and then installing the exact `torch` version required by SadTalker. It also ensures `ffmpeg` is installed, which is crucial for video processing.

In [ ]:
!update-alternatives --install /usr/local/bin/python3 python3 /usr/bin/python3.8 2
!update-alternatives --install /usr/local/bin/python3 python3 /usr/bin/python3.9 1
!sudo apt install python3.8

!sudo apt-get install python3.8-distutils

!python --version

!apt-get update

!apt install software-properties-common

!sudo dpkg --remove --force-remove-reinstreq python3-pip python3-setuptools python3-wheel

!apt-get install python3-pip

print('Git clone project and install requirements...')
!git clone https://github.com/Winfredy/SadTalker &> /dev/null
%cd SadTalker
!export PYTHONPATH=/content/SadTalker:$PYTHONPATH
!python3.8 -m pip install torch==1.12.1+cu113 torchvision==0.13.1+cu113 torchaudio==0.12.1 --extra-index-url https://download.pytorch.org/whl/cu113
!apt update
!apt install ffmpeg &> /dev/null
!python3.8 -m pip install -r requirements.txt

This cell attempts to run the `inference.py` script from SadTalker. The traceback indicates a `ModuleNotFoundError: No module named 'yacs'`, suggesting that not all dependencies were correctly installed, specifically the `yacs` library, which is likely a configuration management library.

In [ ]:
%cd /content/SadTalker
!python inference.py \
  --driven_audio chunk_002.wav \
  --source_image source_image.jpg \
  --result_dir results/ \
  --still_mode \
  --enhancer gfpgan

## First Order Motion Model (FOMM)

First Order Motion Model (FOMM) is another technique for animating a static image using a driving video. This section sets up the environment and downloads pre-trained weights for FOMM.

#FOMM method

In [ ]:
!rm -rf /content/sample_data

This section clones the `first-order-model` repository and installs its dependencies. The error from the previous cell (`4AIEsmXzG877`) indicates that the `%cd first-order-model` was executed *before* the `pip install` commands within the same cell, leading to a `first-order-model` directory not being found. It's important to run `%cd` in a separate cell before dependent commands.

In [ ]:
!rm -rf first-order-model
!git clone https://github.com/AliaksandrSiarohin/first-order-model.git
%cd first-order-model

This cell downloads the pre-trained `vox-cpk.pth.tar` checkpoint file, which is necessary for running the FOMM model. The output shows that the file was successfully downloaded, but its size (88KB) seems unusually small for a model checkpoint, which might indicate an issue with the download or an incorrect file being retrieved.

In [ ]:
!pip install torch torchvision torchaudio --upgrade

%cd first-order-model
!pip install -r requirements.txt

In [ ]:
!mkdir -p checkpoints
!wget -O checkpoints/vox-cpk.pth.tar https://www.dropbox.com/s/64j5hiyq2a1jmt1/vox-cpk.pth.tar?dl=1